# BridgeWatch AI: Model Training

Project: BridgeWatch AI, predicting whether a bridge's condition will deteriorate by its next inspection.
Author: Edmund Goldsberry, UMBC DATA 606 Capstone.

The model uses three data sources: bridge inspection records, weather at each bridge's location, and each
state's spending on bridge repair.

The bridge and weather data cover 2010 to 2025. The spending data only covers 2020 to 2024, so the spending
experiments (Sections 3 to 6) use the 2020 to 2025 records, and the final model (Section 7) is trained on all
16 years.


## 1. Load and join the bridge, weather, and spending data

* `data/processed/bridge_deterioration_dataset_2010.csv.gz` through `bridge_deterioration_dataset_2024.csv.gz`:
  one row per bridge per year to year transition, with the `deteriorated_next_period` label. The number in each
  file name is the `from_year`, the year the row's features come from, so the 15 files hold the transitions
  from 2010 to 11 through 2024 to 25. The table is split by year because the whole thing is too large for a
  single file on GitHub.
* `data/weather_annual_by_bridge_2010.csv` through `data/weather_annual_by_bridge_2025.csv`: one row per
  bridge per calendar year, with annual temperature and precipitation summaries from NOAA. One plain csv file
  per year, so each file stays under GitHub's file size limit.
* `data/sf12a_state_bridge_spending_2020_2024.csv`: one row per state per year (2020 to 2024), with what each
  state spent on bridge replacement and repair, in thousands of dollars, from FHWA Table SF-12A.

The weather data joins on the bridge ID, the state, and the year. The state is part of the key because a structure
number is only unique within a state (the same number appears for both a Michigan and a Texas bridge in 2010). The
spending data only exists at the state level, so it joins on the state and year instead, and every bridge in the same
state and year gets the same value.

Raw spending mostly tells you how big a state is (Texas has about ten times as many bridges as Maryland), so the
spending is turned into repair spending per bridge: the state's repair total divided by its number of
bridges that year.

Spending has not been collected for 2010 to 2019 yet, so transitions before 2020 have no spending value. They are
kept in the data (the bridge and weather features are complete for them) but the spending experiments leave them
out, as Section 3 explains.


In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn plotly joblib streamlit
import sys
print(sys.executable)
print(sys.path)
import http.client
import os
import urllib.error
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/goldiemonster/UMBC-DATA606-Capstone/main"
LOCAL_DATA_DIR = os.path.join("..", "data")
KEY = "STRUCTURE_NUMBER_008"
BRIDGE_YEARS = range(2010, 2025)    # from_year of each bridge file, 2010 to 11 through 2024 to 25
WEATHER_YEARS = range(2010, 2026)
SPENDING_FIRST_YEAR = 2020          # the first year with spending data

# Code columns are read as text, so a code like "5" in one file is never read as 5.0 in another
TEXT_COLS = [
    "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B", "SCOUR_CRITICAL_113", "OWNER_022", "STATE",
    "DECK_STRUCTURE_TYPE_107", "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CULVERT_COND_062",
]
FETCH_ERRORS = (urllib.error.URLError, urllib.error.HTTPError, http.client.HTTPException, OSError)


def load_bridge_data():
    """Load the bridge-year-transition dataset built in eda.ipynb, one file per from_year.

    Tries to fetch all 15 yearly files from GitHub. If any single file fails, the whole attempt is
    abandoned and the local copies are used instead, so a partial GitHub fetch never gets silently mixed
    with local data.
    """
    def read_all(base):
        frames = []
        for year in BRIDGE_YEARS:
            name = f"bridge_deterioration_dataset_{year}.csv.gz"
            frames.append(pd.read_csv(f"{base}/{name}", compression="gzip", dtype={c: str for c in TEXT_COLS},
                                      low_memory=False))
        return pd.concat(frames, ignore_index=True)

    try:
        df = read_all(f"{GITHUB_RAW_BASE}/data/processed")
        print(f"Loaded bridge_deterioration_dataset_{{{min(BRIDGE_YEARS)}..{max(BRIDGE_YEARS)}}}.csv.gz from GitHub")
        return df
    except FETCH_ERRORS as exc:
        local_dir = os.path.join(LOCAL_DATA_DIR, "processed")
        print(f"Could not load all bridge years from GitHub ({exc}); falling back to {local_dir}")
        return read_all(local_dir)


def load_weather_data():
    """Load the annual per-bridge weather dataset, one csv per year, from GitHub or a local copy.

    As with the bridge data, if any single year fails on GitHub the whole attempt is abandoned and the
    local files are used instead.
    """
    def read_all(base):
        return pd.concat([pd.read_csv(f"{base}/weather_annual_by_bridge_{year}.csv") for year in WEATHER_YEARS],
                         ignore_index=True)

    try:
        df = read_all(f"{GITHUB_RAW_BASE}/data")
        print(f"Loaded weather_annual_by_bridge_{{{min(WEATHER_YEARS)}..{max(WEATHER_YEARS)}}}.csv from GitHub")
        return df
    except FETCH_ERRORS as exc:
        print(f"Could not load all weather years from GitHub ({exc}); falling back to {LOCAL_DATA_DIR}")
        return read_all(LOCAL_DATA_DIR)


def join_bridge_and_weather(bridge_df, weather_df):
    """Left join weather onto bridge_df, matching (KEY, STATE, from_year) to (KEY, STATE, YEAR).

    The state is part of the key because a structure number is only unique within a state.
    """
    weather_renamed = weather_df.rename(columns={"YEAR": "weather_year"})
    merged = bridge_df.merge(
        weather_renamed,
        left_on=[KEY, "STATE", "from_year"],
        right_on=[KEY, "STATE", "weather_year"],
        how="left",
    ).drop(columns="weather_year")
    return merged


def load_spending_data():
    """Load the state bridge spending file from GitHub, falling back to a local copy."""
    remote_path = "data/sf12a_state_bridge_spending_2020_2024.csv"
    local_path = os.path.join(LOCAL_DATA_DIR, "sf12a_state_bridge_spending_2020_2024.csv")
    try:
        df = pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}")
        print(f"Loaded {remote_path} from GitHub")
        return df
    except FETCH_ERRORS as exc:
        print(f"Could not load {remote_path} from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path)


def spending_per_bridge(spending_df, bridge_df):
    """Divide each state's yearly repair total by its number of bridges that year."""
    counts = bridge_df.groupby(["STATE", "from_year"]).size().rename("n_bridges").reset_index()
    spending = spending_df.rename(columns={"state": "STATE", "year": "from_year"})
    spending = spending.merge(counts, on=["STATE", "from_year"], how="left")
    spending["repair_per_bridge_thousands"] = spending["bridge_repair_total_thousands"] / spending["n_bridges"]
    return spending[["STATE", "from_year", "repair_per_bridge_thousands"]]


def join_spending(df, spending_df):
    """Left join the per-bridge spending onto df, matching on STATE and from_year."""
    return df.merge(spending_df, on=["STATE", "from_year"], how="left")

/usr/local/bin/python3
['/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/root/.local/lib/python3.11/site-packages', '/usr/local/lib/python3.11/dist-packages', '/usr/lib/python3/dist-packages']


In [2]:
bridge_df = load_bridge_data()
weather_df = load_weather_data()

print(f"Bridge dataset: {bridge_df.shape[0]:,} rows x {bridge_df.shape[1]} columns")
print(f"Weather dataset: {weather_df.shape[0]:,} rows x {weather_df.shape[1]} columns")

df_all = join_bridge_and_weather(bridge_df, weather_df)

assert len(df_all) == len(bridge_df), "Join should not change the row count (weather is one row per bridge-year)"
weather_match_rate = df_all["tavg_mean_c"].notna().mean() * 100

print(f"\nJoined dataset: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")
print(f"Weather match rate: {weather_match_rate:.1f}%")
print("\nTransitions per from_year:")
print(df_all["from_year"].value_counts().sort_index().to_string())
df_all.head()

Loaded bridge_deterioration_dataset_{2010..2024}.csv.gz from GitHub


Loaded weather_annual_by_bridge_{2010..2025}.csv from GitHub
Bridge dataset: 1,985,801 rows x 39 columns
Weather dataset: 2,146,854 rows x 8 columns



Joined dataset: 1,985,801 rows x 44 columns
Weather match rate: 98.6%

Transitions per from_year:
from_year
2010    117129
2011    128876
2012    129814
2013    130382
2014    131318
2015    131911
2016    132584
2017    133469
2018    133905
2019    134572
2020    134869
2021    135587
2022    136309
2023    137156
2024    137920


,STRUCTURE_NUMBER_008,COUNTY_CODE_003,YEAR_BUILT_027,ADT_029,YEAR_ADT_030,PERCENT_ADT_TRUCK_109,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,MAIN_UNIT_SPANS_045,APPR_SPANS_046,STRUCTURE_LEN_MT_049,MAX_SPAN_LEN_MT_048,DECK_WIDTH_MT_052,TRAFFIC_LANES_ON_028A,DECK_STRUCTURE_TYPE_107,SCOUR_CRITICAL_113,OWNER_022,FUNCTIONAL_CLASS_026,DATE_OF_INSPECT_090,INSPECT_FREQ_MONTHS_091,LAT_016,LONG_017,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062,LOWEST_RATING,BRIDGE_CONDITION,STATE,YEAR,LOWEST_RATING_next,BRIDGE_CONDITION_next,from_year,to_year,deteriorated_next_period,age,lat,lon,months_since_inspection,tmax_mean_c,tmin_mean_c,tavg_mean_c,prcp_total_mm,freeze_thaw_days
0,00000000000J003,27.0,1975.0,500.0,2008.0,2.0,2,19,3.0,0.0,6.2,1.8,0.0,2.0,1,6,62,16.0,608.0,24.0,37215400.0,118260200.0,N,N,N,7,7.0,G,CA,2010,7.0,G,2010,2011,0,35.0,37.365000,-118.433889,24.0,22.115925,4.952504,13.534354,147.804688,116.0
1,00000000000J004,27.0,1975.0,500.0,2008.0,2.0,2,19,3.0,0.0,7.7,2.4,0.0,2.0,1,6,62,16.0,608.0,24.0,37215034.0,118254752.0,N,N,N,7,7.0,G,CA,2010,7.0,G,2010,2011,0,35.0,37.363983,-118.429867,24.0,22.115925,4.952504,13.534354,147.804688,116.0
2,00000000000J007,45.0,1978.0,3000.0,2008.0,10.0,3,02,1.0,0.0,18.7,17.3,8.4,2.0,1,6,62,6.0,608.0,24.0,39401800.0,123295400.0,7,8,7,N,7.0,G,CA,2010,7.0,G,2010,2011,0,32.0,39.671667,-123.498333,24.0,19.028018,6.230736,12.629088,1928.250000,15.0
3,00000000000J013,23.0,1962.0,1500.0,2008.0,10.0,2,01,3.0,0.0,25.7,8.5,9.1,2.0,1,6,62,16.0,608.0,24.0,41043000.0,123410000.0,7,7,7,N,7.0,G,CA,2010,7.0,G,2010,2011,0,48.0,41.075000,-123.683333,24.0,20.727269,7.014640,13.871190,1734.242188,7.0
4,00000000000J015,23.0,1966.0,1500.0,2008.0,15.0,2,02,3.0,0.0,34.1,24.4,10.8,2.0,1,6,62,16.0,608.0,24.0,41030600.0,123410000.0,6,7,7,N,6.0,F,CA,2010,6.0,F,2010,2011,0,44.0,41.051667,-123.683333,24.0,20.727269,7.014640,13.871190,1734.242188,7.0


Next, add the spending data. The spending year is matched to `from_year`, the year each row's features come
from, so a bridge's 2022 record gets its state's 2022 spending. The spending file covers 2020 to 2024, so the
rows with a `from_year` of 2020 or later should all find a match. The earlier rows get no spending value, and
they are left empty on purpose rather than filled in with a guess.

In [3]:
spending_df = load_spending_data()
per_bridge = spending_per_bridge(spending_df, bridge_df)

rows_before = len(df_all)
df_all = join_spending(df_all, per_bridge)

assert len(df_all) == rows_before, "Join should not change the row count (spending is one row per state-year)"
in_spending_era = df_all["from_year"] >= SPENDING_FIRST_YEAR
spending_match_rate = df_all.loc[in_spending_era, "repair_per_bridge_thousands"].notna().mean() * 100
print(f"Spending match rate for rows from {SPENDING_FIRST_YEAR} on: {spending_match_rate:.1f}% of {in_spending_era.sum():,}")
print(f"Rows before {SPENDING_FIRST_YEAR} (no spending data): {(~in_spending_era).sum():,}")

print("\nRepair spending per bridge, thousands of dollars:")
print(per_bridge.pivot(index="STATE", columns="from_year", values="repair_per_bridge_thousands").round(1))

Loaded data/sf12a_state_bridge_spending_2020_2024.csv from GitHub


Spending match rate for rows from 2020 on: 100.0% of 681,841
Rows before 2020 (no spending data): 1,303,960

Repair spending per bridge, thousands of dollars:
from_year  2020  2021  2022  2023  2024
STATE                                  
CA          7.7   9.0   9.3  14.3  15.9
FL         24.5  33.4  41.8  41.8  39.7
MD         35.2  29.4  22.9  35.1  24.8
MI         51.4  52.9  56.1  53.3  66.9
NY         61.7  49.8  51.2  51.1  38.5
TX         14.7  14.1  14.2  22.6  25.8
WA         26.2  56.6  75.7  33.5  37.4


## 2. Define and clean the feature set

The model predicts `deteriorated_next_period` (1 if a bridge's condition rating dropped by the next
inspection, 0 otherwise). The candidate features fall into two groups:

* Categorical columns, encoded as one-hot indicators (material type, ownership, condition ratings, and so
  on). These get cast to text first, since a couple of them mix numeric-looking codes with the letter `N`
  for "not applicable", and mixing types can confuse the encoder later.
* Numeric columns, which get scaled before modeling (age, traffic volume, span length, the weather
  features, and repair spending per bridge, among others).

A few columns are left out on purpose: `STRUCTURE_NUMBER_008` is an identifier, not a predictor.
`YEAR_BUILT_027` is replaced by `age`, which is the more directly useful version of the same information.
`BRIDGE_CONDITION` and the year columns (`from_year`, `to_year`, `YEAR`) are either derived from the label or
used only for the train/test split, not fed to the model.

In [4]:
TARGET = "deteriorated_next_period"

CATEGORICAL_COLS = [
    "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B", "SCOUR_CRITICAL_113", "OWNER_022", "STATE",
    "DECK_STRUCTURE_TYPE_107", "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CULVERT_COND_062",
]

NUMERIC_COLS = [
    "age", "ADT_029", "PERCENT_ADT_TRUCK_109", "MAIN_UNIT_SPANS_045", "STRUCTURE_LEN_MT_049",
    "DECK_WIDTH_MT_052", "TRAFFIC_LANES_ON_028A", "INSPECT_FREQ_MONTHS_091", "LOWEST_RATING", "lat", "lon",
    "tmax_mean_c", "tmin_mean_c", "tavg_mean_c", "prcp_total_mm", "freeze_thaw_days",
    "repair_per_bridge_thousands",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

assert set(CATEGORICAL_COLS) == set(TEXT_COLS), "the columns read as text in Section 1 should be the categorical ones"

print(f"{len(CATEGORICAL_COLS)} categorical columns, {len(NUMERIC_COLS)} numeric columns, "
      f"{len(FEATURE_COLS)} features total")

10 categorical columns, 17 numeric columns, 27 features total


In [5]:
def clean_features(df, categorical_cols=CATEGORICAL_COLS):
    """Cast categorical columns to text and turn blank strings into real missing values."""
    df = df.copy()
    for col in categorical_cols:
        df[col] = df[col].astype(str).replace("", np.nan)
    return df

In [6]:
df_all = clean_features(df_all)

print("Missing values by feature (percent):")
print((df_all[FEATURE_COLS].isna().mean() * 100).round(2).sort_values(ascending=False))

print(f"\nTarget balance: {df_all[TARGET].value_counts(normalize=True).round(4).to_dict()}")

Missing values by feature (percent):
repair_per_bridge_thousands    65.66
PERCENT_ADT_TRUCK_109           7.74
freeze_thaw_days                1.36
tavg_mean_c                     1.36
tmin_mean_c                     1.36
prcp_total_mm                   1.36
tmax_mean_c                     1.36
lat                             0.03
SCOUR_CRITICAL_113              0.03
lon                             0.03
age                             0.01
CULVERT_COND_062                0.00
SUBSTRUCTURE_COND_060           0.00
SUPERSTRUCTURE_COND_059         0.00
DECK_COND_058                   0.00
STATE                           0.00
DECK_STRUCTURE_TYPE_107         0.00
OWNER_022                       0.00
STRUCTURE_TYPE_043B             0.00
STRUCTURE_KIND_043A             0.00
LOWEST_RATING                   0.00
TRAFFIC_LANES_ON_028A           0.00
INSPECT_FREQ_MONTHS_091         0.00
STRUCTURE_LEN_MT_049            0.00
MAIN_UNIT_SPANS_045             0.00
ADT_029                         0.00
D

## 3. Split into train and test sets

The split is based on time, not a random shuffle. The model trains on every transition that ends before 2025,
and gets evaluated on the transitions ending in 2025, which it never sees during training. This mirrors how the
model would actually be used: forecasting next year's risk for bridges that are already being tracked, using
only information that would have been available before that forecast.

The same bridge can show up in both the training set (its 2022 to 23 transition, for example) and the test
set (its 2024 to 25 transition). That is not label leakage. The model never sees a bridge's 2025 outcome while
training, since only earlier transitions are in the training set.

Sections 3 to 6 are about the spending data, which only exists from 2020. To compare models with and without
spending on equal terms, these sections use only the transitions from 2020 on (`df`), so training covers
transitions ending 2021 to 2024. Section 7 goes back to all 16 years (`df_all`) for the final model, which does
not use spending. The test set is the same in both cases: the transitions ending in 2025.

The spending data follows the same timeline. Training rows use spending from 2020 to 2023, and test rows use
2024 spending, which is the most recent year that would be known before the 2025 inspections.


In [7]:
def time_based_split(df, test_year=2025):
    """Train on every to_year before test_year; test on to_year == test_year."""
    train = df[df["to_year"] < test_year].copy()
    test = df[df["to_year"] == test_year].copy()
    return train, test

In [8]:
df = df_all[df_all["from_year"] >= SPENDING_FIRST_YEAR].copy()
print(f"Spending era: {len(df):,} of {len(df_all):,} transitions have a from_year of {SPENDING_FIRST_YEAR} or later\n")

train_df, test_df = time_based_split(df)

assert len(train_df) + len(test_df) == len(df), "train and test should cover every row exactly once"
assert set(train_df["to_year"]).isdisjoint(set(test_df["to_year"])), "train and test years should not overlap"

print(f"Train: {len(train_df):,} rows, to_year in {sorted(train_df['to_year'].unique())}")
print(f"Test:  {len(test_df):,} rows, to_year in {sorted(test_df['to_year'].unique())}")

print(f"\nTrain positive rate: {100 * train_df[TARGET].mean():.2f}%")
print(f"Test positive rate:  {100 * test_df[TARGET].mean():.2f}% ({test_df[TARGET].sum():,} positives)")

print(f"\nSpending years used in train: {sorted(train_df['from_year'].unique())}")
print(f"Spending years used in test:  {sorted(test_df['from_year'].unique())}")

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

Spending era: 681,841 of 1,985,801 transitions have a from_year of 2020 or later



Train: 543,921 rows, to_year in [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Test:  137,920 rows, to_year in [np.int64(2025)]

Train positive rate: 4.29%
Test positive rate:  4.54% (6,268 positives)

Spending years used in train: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Spending years used in test:  [np.int64(2024)]


## 4. Build and fit the preprocessing and model pipeline

The model is a Logistic Regression, chosen for interpretability: its coefficients can be read directly as
"this factor pushes risk up" or "this factor pushes risk down", which is what the app needs to show for each
bridge.

Before the model can use the data, it needs two kinds of preprocessing:

* Numeric columns get missing values filled with the column median, then scaled to have a mean of 0 and a
  standard deviation of 1. Scaling matters for Logistic Regression because the size of each coefficient
  depends on the scale of its input; without it, a feature like `prcp_total_mm` (values in the hundreds)
  would dominate a feature like `age` (values in the tens) for reasons that have nothing to do with actual
  importance.
* Categorical columns get missing values filled with the most frequent category, then one hot encoded (turned
  into a set of 0/1 indicator columns, one per category).

Both steps are combined into a single scikit-learn pipeline, fit only on the training set, so nothing about
the test set leaks into how the data gets transformed.

Class imbalance is handled with `class_weight="balanced"`, which tells Logistic Regression to weight the rare
"deteriorated" class more heavily during training, rather than resampling the data.

Repair spending per bridge is scaled like every other numeric feature.

In [9]:

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_pipeline():
    """Preprocessing (impute + scale/encode) plus a balanced Logistic Regression."""
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ])
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    return Pipeline([("preprocess", preprocessor), ("clf", model)])

In [10]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

n_encoded_features = len(pipeline.named_steps["preprocess"].get_feature_names_out())
print(f"Fit complete. {len(FEATURE_COLS)} raw features expand to {n_encoded_features} "
      f"after scaling and one-hot encoding.")

Fit complete. 27 raw features expand to 154 after scaling and one-hot encoding.


## 5. Evaluate performance on the 2025 test set

The test set is the 137,920 bridge transitions ending in 2025 that the model never trained on (see
Section 3), so these numbers reflect how the model would perform forecasting next year's risk for bridges it
already knows about.

Accuracy is not a useful headline metric here: only 4.5% of test rows actually deteriorated, so a model that
always predicts "no" would already be about 95% accurate while catching zero at-risk bridges. Instead this
section leads with ROC AUC and average precision (area under the precision-recall curve), which are not
fooled by the imbalance, then looks at precision and recall at the default 0.5 threshold, and finally checks
how well the model's risk ranking would work if it were used to prioritize which bridges get inspected first.

In [11]:
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix, roc_auc_score,
)

train_proba = pipeline.predict_proba(X_train)[:, 1]
test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print(f"Train ROC AUC:            {roc_auc_score(y_train, train_proba):.3f}")
print(f"Test ROC AUC:             {roc_auc_score(y_test, test_proba):.3f}")
print(f"Train average precision:  {average_precision_score(y_train, train_proba):.3f}")
print(f"Test average precision:   {average_precision_score(y_test, test_proba):.3f}")
print(f"(Test positive rate, i.e. the average-precision floor a random ranking would score: "
      f"{y_test.mean():.3f})")

print("\nClassification report on the 2025 test set, at the default 0.5 probability threshold:")
print(classification_report(
    y_test, test_pred, target_names=["not deteriorated", "deteriorated"], digits=3
))

cm = confusion_matrix(y_test, test_pred)
cm_df = pd.DataFrame(
    cm,
    index=["actual: not deteriorated", "actual: deteriorated"],
    columns=["predicted: not deteriorated", "predicted: deteriorated"],
)
print("Confusion matrix:")
print(cm_df)

Train ROC AUC:            0.722
Test ROC AUC:             0.716


Train average precision:  0.111
Test average precision:   0.114
(Test positive rate, i.e. the average-precision floor a random ranking would score: 0.045)

Classification report on the 2025 test set, at the default 0.5 probability threshold:
                  precision    recall  f1-score   support

not deteriorated      0.975     0.663     0.790    131652
    deteriorated      0.083     0.642     0.147      6268

        accuracy                          0.662    137920
       macro avg      0.529     0.653     0.469    137920
    weighted avg      0.934     0.662     0.760    137920

Confusion matrix:
                          predicted: not deteriorated  predicted: deteriorated
actual: not deteriorated                        87341                    44311
actual: deteriorated                             2242                     4026


The classification report above is a single snapshot at the 0.5 threshold. Because class_weight="balanced"
was used, that threshold is not necessarily the right operating point for this problem, it treats a missed
deterioration and a false alarm as equally costly, which is unlikely to be true for a bridge inspection
program. The ROC and precision-recall curves below show the full tradeoff across every threshold at once.

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve, roc_curve

MODEL_COLOR = "#2a78d6"
REFERENCE_COLOR = "#c3c2b7"

fpr, tpr, _ = roc_curve(y_test, test_proba)
precision, recall, _ = precision_recall_curve(y_test, test_proba)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("ROC curve", "Precision-recall curve"),
)

fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode="lines", name="model", line=dict(color=MODEL_COLOR, width=2)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="random guess",
               line=dict(color=REFERENCE_COLOR, width=2, dash="dash")),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=recall, y=precision, mode="lines", name="model", showlegend=False,
               line=dict(color=MODEL_COLOR, width=2)),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[y_test.mean(), y_test.mean()], mode="lines", name="random guess",
               showlegend=False, line=dict(color=REFERENCE_COLOR, width=2, dash="dash")),
    row=1, col=2,
)

fig.update_xaxes(title_text="False positive rate", range=[0, 1], gridcolor="#e1e0d9", row=1, col=1)
fig.update_yaxes(title_text="True positive rate", range=[0, 1], gridcolor="#e1e0d9", row=1, col=1)
fig.update_xaxes(title_text="Recall", range=[0, 1], gridcolor="#e1e0d9", row=1, col=2)
fig.update_yaxes(title_text="Precision", range=[0, 1], gridcolor="#e1e0d9", row=1, col=2)
fig.update_layout(
    title_text="Model performance on the 2025 held-out test set",
    height=450, width=950,
    plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
)
fig.show()

### Capture rate: how useful is the model for prioritizing inspections?

BridgeWatch AI's actual use case is ranking bridges by risk, not making a one-off yes/no call, so the most
relevant test is: if inspectors worked down the model's risk ranking, how many of the bridges that truly
deteriorated in 2025 would they find within the first X% of bridges checked? This "capture rate" is a more
direct measure of real-world usefulness than any single classification metric above.

In [13]:
def capture_at_top_k(y_true, y_score, k_frac):
    """Share of actual positives found within the top k_frac of rows by predicted score."""
    y_true = np.asarray(y_true)
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    top_k_idx = np.argsort(-y_score)[:k]
    return y_true[top_k_idx].sum() / y_true.sum(), k


y_test_arr = y_test.to_numpy()
n_positive = int(y_test_arr.sum())

print(f"{n_positive:,} of {len(y_test_arr):,} test-set bridges actually deteriorated in 2025.\n")
print(f"{'Top % of bridges by risk':<28}{'# bridges':>12}{'% of deteriorations found':>28}")
for k_frac in [0.05, 0.10, 0.20, 0.30, 0.50]:
    capture_rate, k = capture_at_top_k(y_test_arr, test_proba, k_frac)
    print(f"{int(k_frac * 100):>3}%{'':<24}{k:>12,}{capture_rate * 100:>27.1f}%")

6,268 of 137,920 test-set bridges actually deteriorated in 2025.

Top % of bridges by risk       # bridges   % of deteriorations found
  5%                               6,896                       17.6%
 10%                              13,792                       30.3%
 20%                              27,584                       47.8%
 30%                              41,376                       59.3%
 50%                              68,960                       78.4%


## 6. Do the weather and spending data help?

The model in Section 4 uses all three data sources. To see what each one adds, I will fit the same model three ways
and score each version on the same 2025 test set:

1. Bridge only: just the bridge inspection features.
2. Bridge + weather: adds the five weather features (research question 3).
3. Bridge + weather + spending: adds repair spending per bridge. This is the model from Section 4
   (research question 4).

These comparisons use the 2020 to 2025 records, since that is where all three data sources exist.

Everything else stays the same: same preprocessing, same Logistic Regression settings, same train and test rows.
So any change in the scores comes from the added data, and nothing else. Each version is scored four ways: ROC AUC,
average precision, and the share of 2025 deteriorations found in the top 10% and top 20% of bridges by risk
(the capture rate from Section 5).

In [14]:
from sklearn.metrics import average_precision_score, roc_auc_score

WEATHER_COLS = ["tmax_mean_c", "tmin_mean_c", "tavg_mean_c", "prcp_total_mm", "freeze_thaw_days"]
SPENDING_COLS = ["repair_per_bridge_thousands"]
BRIDGE_NUMERIC_COLS = [c for c in NUMERIC_COLS if c not in WEATHER_COLS + SPENDING_COLS]


def build_pipeline_for(numeric_cols):
    """Same preprocessing and model as build_pipeline(), but with a chosen list of numeric columns."""
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ])
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    return Pipeline([("preprocess", preprocessor), ("clf", model)])


def score_model(y_true, y_score):
    """The four test-set scores used to compare model versions."""
    return {
        "ROC AUC": roc_auc_score(y_true, y_score),
        "Average precision": average_precision_score(y_true, y_score),
        "Found in top 10%": capture_at_top_k(y_true, y_score, 0.10)[0],
        "Found in top 20%": capture_at_top_k(y_true, y_score, 0.20)[0],
    }

In [15]:
versions = {
    "Bridge only": BRIDGE_NUMERIC_COLS,
    "Bridge + weather": BRIDGE_NUMERIC_COLS + WEATHER_COLS,
}

test_scores = {}
for name, numeric_cols in versions.items():
    cols = numeric_cols + CATEGORICAL_COLS
    version_pipeline = build_pipeline_for(numeric_cols)
    version_pipeline.fit(train_df[cols], y_train)
    test_scores[name] = version_pipeline.predict_proba(test_df[cols])[:, 1]

# The full model from Section 4 already has its test scores
test_scores["Bridge + weather + spending"] = test_proba

comparison = pd.DataFrame({name: score_model(y_test_arr, s) for name, s in test_scores.items()}).T
print("Scores on the 2025 test set (higher is better for all four):")
print(comparison.round(4))

Scores on the 2025 test set (higher is better for all four):
                             ROC AUC  Average precision  Found in top 10%  Found in top 20%
Bridge only                   0.7213             0.1172            0.3109            0.4874
Bridge + weather              0.7216             0.1170            0.3130            0.4892
Bridge + weather + spending   0.7159             0.1141            0.3034            0.4781


### How sure are we about these differences?

The differences between versions are small, so it is fair to ask whether they are real or just luck in which
bridges happened to land in the test set. A simple way to check is bootstrapping: redraw the test set at
random 200 times, recompute each version's ROC AUC on every redraw, and look at how much
the gain from adding each data source moves around. If the middle 95% of those gains stays on one side of zero,
the difference is very unlikely to be luck.

In [16]:
rng = np.random.default_rng(42)
n_test = len(y_test_arr)
gains = {"Adding weather": [], "Adding spending": []}

for _ in range(200):
    idx = rng.integers(0, n_test, n_test)
    auc = {name: roc_auc_score(y_test_arr[idx], s[idx]) for name, s in test_scores.items()}
    gains["Adding weather"].append(auc["Bridge + weather"] - auc["Bridge only"])
    gains["Adding spending"].append(auc["Bridge + weather + spending"] - auc["Bridge + weather"])

for name, values in gains.items():
    low, high = np.percentile(values, [2.5, 97.5])
    print(f"{name:<16} ROC AUC change: {np.mean(values):+.4f}  (95% of redraws between {low:+.4f} and {high:+.4f})")

Adding weather   ROC AUC change: +0.0003  (95% of redraws between -0.0003 and +0.0009)
Adding spending  ROC AUC change: -0.0057  (95% of redraws between -0.0063 and -0.0051)


The results split in two.

Weather makes no clear difference. Adding the weather features raises ROC AUC by only 0.0003, and the middle 95% of
redraws runs from slightly below zero to slightly above it, so the gain cannot be told apart from luck. The share of
deteriorations found in the top 10% and 20% moves up by about 0.2 points, and average precision is flat. So weather
neither helps nor hurts here (research question 3).

Spending makes the model worse, and that is real. Adding spending per bridge lowers ROC AUC by about 0.006, and the
model finds fewer of the bridges that actually deteriorated in its top 10% and 20%. The whole 95% range is below
zero, so this is not bad luck.


In [17]:
state_year = (df.groupby(["STATE", "from_year"])
              .agg(spending=("repair_per_bridge_thousands", "first"),
                   deterioration_rate=(TARGET, "mean"))
              .reset_index())

# Training years only: in years a state spent more than its usual, did it deteriorate more or less?
train_years = state_year[state_year["from_year"] < 2024].copy()
for col in ["spending", "deterioration_rate"]:
    train_years[col + "_vs_avg"] = train_years[col] - train_years.groupby("STATE")[col].transform("mean")
train_corr = train_years["spending_vs_avg"].corr(train_years["deterioration_rate_vs_avg"])
print(f"Within-state correlation in the training years (2020 to 2023): {train_corr:+.2f}")

# The test year: how did each state's 2024 spending and deterioration compare with its 2020 to 2023 average?
train_avg = train_years.groupby("STATE")[["spending", "deterioration_rate"]].mean()
year_2024 = state_year[state_year["from_year"] == 2024].set_index("STATE")
change = pd.DataFrame({
    "2024 spending vs usual ($K per bridge)": (year_2024["spending"] - train_avg["spending"]).round(1),
    "2024 deterioration vs usual (% points)": ((year_2024["deterioration_rate"]
                                                - train_avg["deterioration_rate"]) * 100).round(2),
})
print("\nHow 2024 compared with each state's 2020 to 2023 average:")
print(change)

Within-state correlation in the training years (2020 to 2023): +0.21

How 2024 compared with each state's 2020 to 2023 average:
       2024 spending vs usual ($K per bridge)  2024 deterioration vs usual (% points)
STATE                                                                                
CA                                        5.9                                   -0.57
FL                                        4.3                                   -1.26
MD                                       -5.9                                   -0.74
MI                                       13.5                                   -1.40
NY                                      -14.9                                    5.38
TX                                        9.4                                   -0.12
WA                                      -10.6                                   -0.11


This shows what went wrong. In the training years, years when a state spent more than usual had slightly
*more* deterioration (a correlation of about +0.2), so the model learned that higher spending means higher risk.
In 2024 that pattern turned around. New York spent about $15,000 less per bridge than usual, and its
deterioration jumped by more than 5 percentage points. Michigan spent the most extra and saw its deterioration
fall. The model used the rule it learned, scored New York's bridges as lower risk in exactly the year they got
worse, and ranked them badly.

The underlying problem is that spending has only 35 values (7 states by 5 years), and the model learns from just
28 of them. That is not enough to find a pattern that holds from one year to the next, and this one did not.

What this means for the project:

* Research question 4: in this data, state bridge spending does not help predict which bridges will
  deteriorate. Across states, higher spending goes with more deterioration because states spend where bridges
  are already in poor shape (EDA Section 9). Within states, the year to year link is too weak and unstable to
  forecast with.
* Research question 3: once a bridge's own features and its state are known, weather adds almost nothing. That fits
  the EDA, where the climate differences between states (Section 8.2) are already captured by the `STATE` feature.
* The final model: spending is dropped. Weather stays, because it does no harm and the project asks about it.
  Spending can still be shown in the app as background about each state, just not used to score bridges.


## 7. The final model: bridge and weather features, trained on more years

Spending did not help, so the final model drops it. That also removes the reason for the 2020 cutoff: the bridge
and weather features exist for every year back to 2010, so the final model could train on all of them. Whether it
should depends on whether the older years teach the model things that still hold today, and Section 7.1 tests
that. The test set stays the same (the transitions ending in 2025), so the final model can be compared directly
with everything above.


### 7.1 How much history should the model train on?

The EDA (Section 5) found that California's ratings swung sharply in 2016 to 2020: a wave of upgrades in 2016,
then 23% and 25% of bridges dropping in 2017 and 2018. That looks like a change in how the ratings were assigned,
not bridges wearing out, and training on it could teach the model the wrong lessons. Three choices of training
data are compared:

1. All years (2010 on).
2. All years, but leaving out California transitions that end in 2016 to 2020.
3. Only 2020 on, the window used in Sections 3 to 6.

The 2025 test set should not be used to make this choice, or the final scores would be too optimistic. So the
comparison uses 2024 as a validation year instead: each choice trains on transitions ending before 2024 and is
scored on the transitions ending in 2024. The choice that does best there is used for the final model. If two
choices are within 0.001 of each other in ROC AUC, they count as tied, and the one with more training data is used.


In [18]:
FINAL_NUMERIC_COLS = BRIDGE_NUMERIC_COLS + WEATHER_COLS
FINAL_FEATURE_COLS = FINAL_NUMERIC_COLS + CATEGORICAL_COLS

assert "repair_per_bridge_thousands" not in FINAL_FEATURE_COLS, "the final model should not use spending"
print(f"Final model: {len(CATEGORICAL_COLS)} categorical and {len(FINAL_NUMERIC_COLS)} numeric features, "
      f"{len(FINAL_FEATURE_COLS)} total (was {len(FEATURE_COLS)} with spending)")

VALIDATION_YEAR = 2024


def california_swing(d):
    """California transitions that end in 2016 to 2020, when its ratings swung sharply (EDA Section 5)."""
    return (d["STATE"] == "CA") & d["to_year"].between(2016, 2020)


TRAINING_CHOICES = {
    "All years (2010 on)": lambda d: d,
    "All years, without California 2016 to 2020": lambda d: d[~california_swing(d)],
    "2020 on only": lambda d: d[d["from_year"] >= SPENDING_FIRST_YEAR],
}


def fit_final_model(train_rows):
    """Fit the bridge + weather model on the given rows."""
    pipe = build_pipeline_for(FINAL_NUMERIC_COLS)
    pipe.fit(train_rows[FINAL_FEATURE_COLS], train_rows[TARGET])
    return pipe


val_train, val_rows = time_based_split(df_all, test_year=VALIDATION_YEAR)
y_val = val_rows[TARGET].to_numpy()

validation_scores = {}
for name, choose in TRAINING_CHOICES.items():
    rows = choose(val_train)
    pipe = fit_final_model(rows)
    validation_scores[name] = {"Training rows": len(rows),
                               **score_model(y_val, pipe.predict_proba(val_rows[FINAL_FEATURE_COLS])[:, 1])}

validation_table = pd.DataFrame(validation_scores).T
print(f"Validation year {VALIDATION_YEAR} (each choice trained on transitions ending before {VALIDATION_YEAR}):")
print(validation_table.round(4).astype({"Training rows": int}))

# Choices within 0.001 ROC AUC of the best count as tied, and a tie goes to the choice with more training rows
best_auc = validation_table["ROC AUC"].max()
tied = validation_table[validation_table["ROC AUC"] >= best_auc - 0.001]
best_choice = tied["Training rows"].idxmax()
print(f"\nBest on the validation year (ties go to the larger training set): {best_choice}")

Final model: 10 categorical and 16 numeric features, 26 total (was 27 with spending)


Validation year 2024 (each choice trained on transitions ending before 2024):
                                            Training rows  ROC AUC  Average precision  Found in top 10%  Found in top 20%
All years (2010 on)                               1710725   0.6868             0.0891            0.2692            0.4218
All years, without California 2016 to 2020        1583562   0.7025             0.0949            0.2824            0.4384
2020 on only                                       406765   0.7024             0.0987            0.2933            0.4377

Best on the validation year (ties go to the larger training set): All years, without California 2016 to 2020


On the validation year, the first choice (all years, nothing removed) does clearly worst, with a ROC AUC of 0.687
against about 0.702 for the other two. Leaving out California's 2016 to 2020 transitions is what fixes it: with
those rows removed, the model matches the 2020 on model (0.7025 against 0.7024) while training on almost four times
as much data. So the older years are not the problem. California's swing years are.

The two best choices are tied, so the rule from above applies and the larger training set wins: all years, without
California 2016 to 2020. The 2025 test set has not been used to make this choice.


### 7.2 Fit the final model

The final model uses the best choice from 7.1, trained on every transition ending before 2025, and is scored on the 2025
test set. The other two choices are fit as well so the test scores can be compared side by side.


In [19]:
full_train, final_test_df = time_based_split(df_all)
assert final_test_df.index.equals(test_df.index), "the test set should not change"

choice_pipelines, choice_test_proba = {}, {}
for name, choose in TRAINING_CHOICES.items():
    rows = choose(full_train)
    choice_pipelines[name] = fit_final_model(rows)
    choice_test_proba[name] = choice_pipelines[name].predict_proba(test_df[FINAL_FEATURE_COLS])[:, 1]
    if name == best_choice:
        final_train_df = rows

final_pipeline = choice_pipelines[best_choice]
final_test_proba = choice_test_proba[best_choice]
final_y_train = final_train_df[TARGET]

# The 2020 on choice is the same model as "Bridge + weather" in Section 6, so its scores should match
assert np.allclose(choice_test_proba["2020 on only"], test_scores["Bridge + weather"], atol=1e-6)

print(f"Final model: {best_choice}")
print(f"Train: {len(final_train_df):,} rows, to_year {final_train_df['to_year'].min()} to {final_train_df['to_year'].max()} "
      f"(Section 4 used {len(train_df):,})")
print(f"Test:  {len(final_test_df):,} rows, the same as before")

n_encoded = len(final_pipeline.named_steps["preprocess"].get_feature_names_out())
print(f"Fit complete. {len(FINAL_FEATURE_COLS)} raw features expand to {n_encoded} after scaling and one-hot encoding.")

Final model: All years, without California 2016 to 2020
Train: 1,720,718 rows, to_year 2011 to 2024 (Section 4 used 543,921)
Test:  137,920 rows, the same as before
Fit complete. 26 raw features expand to 154 after scaling and one-hot encoding.


### Final scores on the 2025 test set

The table below puts the three training choices next to the Section 4 model (which included spending and trained on
2020 on). The "2020 on only" row is the "Bridge + weather" model from Section 6.


In [20]:
comparison_rows = {"Section 4 model (bridge + weather + spending, 2020 on)": score_model(y_test_arr, test_proba)}
for name, proba in choice_test_proba.items():
    label = f"Bridge + weather: {name}" + (" (final)" if name == best_choice else "")
    comparison_rows[label] = score_model(y_test_arr, proba)
final_comparison = pd.DataFrame(comparison_rows).T
print(final_comparison.round(4))

print(f"\nFinal model, {best_choice}:")
print(f"{'Top % of bridges by risk':<28}{'# bridges':>12}{'% of deteriorations found':>28}")
for k_frac in [0.05, 0.10, 0.20, 0.30, 0.50]:
    capture_rate, k = capture_at_top_k(y_test_arr, final_test_proba, k_frac)
    print(f"{int(k_frac * 100):>3}%{'':<24}{k:>12,}{capture_rate * 100:>27.1f}%")

                                                    ROC AUC  Average precision  Found in top 10%  Found in top 20%
Section 4 model (bridge + weather + spending, 2...   0.7159             0.1141            0.3034            0.4781
Bridge + weather: All years (2010 on)                0.7134             0.1130            0.2837            0.4820
Bridge + weather: All years, without California...   0.7259             0.1202            0.3093            0.5034
Bridge + weather: 2020 on only                       0.7216             0.1170            0.3130            0.4892

Final model, All years, without California 2016 to 2020:
Top % of bridges by risk       # bridges   % of deteriorations found
  5%                               6,896                       18.6%
 10%                              13,792                       30.9%
 20%                              27,584                       50.3%
 30%                              41,376                       61.8%
 50%                  

### How sure are we about the final model's gain?

Same idea as in Section 6: redraw the test set 200 times and look at how much the final model's ROC AUC advantage moves around.


In [21]:
rng = np.random.default_rng(42)
comparisons = {"vs. 2020 on only": "2020 on only", "vs. all years with nothing removed": "All years (2010 on)"}
gains = {label: [] for label in comparisons}

for _ in range(200):
    idx = rng.integers(0, n_test, n_test)
    final_auc = roc_auc_score(y_test_arr[idx], final_test_proba[idx])
    for label, other in comparisons.items():
        gains[label].append(final_auc - roc_auc_score(y_test_arr[idx], choice_test_proba[other][idx]))

for label, values in gains.items():
    low, high = np.percentile(values, [2.5, 97.5])
    print(f"Final model {label:<38} ROC AUC change: {np.mean(values):+.4f}  (95% of redraws between {low:+.4f} and {high:+.4f})")

Final model vs. 2020 on only                       ROC AUC change: +0.0042  (95% of redraws between +0.0025 and +0.0062)
Final model vs. all years with nothing removed     ROC AUC change: +0.0126  (95% of redraws between +0.0107 and +0.0146)


On the 2025 test set the final model scores a ROC AUC of 0.726, against 0.722 for the model trained on 2020 on and
0.713 for all years with nothing removed. It also finds more of the deteriorations in its top 20% (50.3% against
48.9%), and average precision rises from 0.117 to 0.120. In the top 10% it is slightly behind the 2020 on model (30.9%
against 31.3%), so the gain comes from the wider ranking and not from the very top.

The bootstrap says the gain over the 2020 on model is real: ROC AUC is higher by about 0.004, and the middle 95% of
redraws runs from +0.0025 to +0.0062, all above zero. The gain over training on all years with nothing removed is
larger (about +0.013), which shows how much California's 2016 to 2020 swing hurts.

So the extra years do help, but only once the unusual California years are left out. In absolute terms the gain is
small, as with every comparison in this notebook, because a bridge's own condition rating does most of the work.


### How the model got here

| Stage | What was tried | Test ROC AUC | Outcome |
|---|---|---|---|
| Starting point | Bridge inspection features only, trained on 2020 on | 0.721 | Baseline |
| Add weather (research question 3) | Temperature, precipitation, and freeze and thaw days at each bridge | 0.722 | No clear gain, no harm, kept |
| Add spending (research question 4) | State bridge repair spending per bridge | 0.716 | Made predictions worse, dropped |
| Train on all years | Add the 2010 to 2019 transitions | 0.713 | Worse, because of California's 2016 to 2020 rating swing |
| Leave out California 2016 to 2020 | All years except those California transitions | 0.726 | Best, used for the app |

The spending data is not thrown away. The EDA (Section 9 of eda.ipynb) keeps its analysis of how spending relates
to deterioration across and within states, and the app can still show each state's spending as background
information. It just is not used to score bridges. The California years are not thrown away either: they stay in the
processed data files, and the EDA (Section 5) shows them. They are only left out when training the final model.

The next step looks inside the final model: which features push a bridge's risk up or down, and by how much.


## 8. What drives the model's risk scores?

Research question 2 asks which factors are the strongest predictors of deterioration. Logistic Regression answers
this directly through its coefficients, one per feature (after one hot encoding, so one per category too).

How to read them:

* A positive coefficient raises the predicted risk; a negative one lowers it.
* The odds multiplier (the coefficient run through `exp()`) is easier to read. A multiplier of 1.25 means the
  odds of deteriorating are 25% higher; 0.80 means they are 20% lower.
* For a numeric feature, the multiplier is for a one standard deviation increase, since the features were
  scaled in Section 4. The table below converts that back to real units (for example, years of age).
* For a category (for example, a superstructure rated 3), the multiplier compares that category with the
  other categories of the same column.

In [22]:
final_preprocess = final_pipeline.named_steps["preprocess"]
final_clf = final_pipeline.named_steps["clf"]


def readable_name(encoded_name):
    """Turn 'num__age' into 'age' and 'cat__DECK_COND_058_5' into 'DECK_COND_058 = 5'."""
    kind, feature = encoded_name.split("__", 1)
    if kind == "num":
        return feature
    for col in CATEGORICAL_COLS:
        if feature.startswith(col + "_"):
            return f"{col} = {feature[len(col) + 1:]}"
    return feature


coef_table = pd.DataFrame({
    "encoded_name": final_preprocess.get_feature_names_out(),
    "coefficient": final_clf.coef_[0],
})
coef_table["feature"] = coef_table["encoded_name"].map(readable_name)
coef_table["odds_multiplier"] = np.exp(coef_table["coefficient"])
coef_table["kind"] = np.where(coef_table["encoded_name"].str.startswith("num__"), "numeric", "category")
print(f"{len(coef_table)} coefficients: {(coef_table['kind'] == 'numeric').sum()} numeric, "
      f"{(coef_table['kind'] == 'category').sum()} category indicators")

154 coefficients: 16 numeric, 138 category indicators


### 8.1 Numeric features

In [23]:
scaler = final_preprocess.named_transformers_["num"].named_steps["scale"]

numeric_effects = coef_table[coef_table["kind"] == "numeric"].copy()
numeric_effects["one_sd_in_real_units"] = scaler.scale_
numeric_effects = numeric_effects.sort_values("odds_multiplier", ascending=False)

print("Odds multiplier for a one standard deviation increase in each numeric feature:")
print(numeric_effects[["feature", "one_sd_in_real_units", "odds_multiplier"]]
      .round({"one_sd_in_real_units": 2, "odds_multiplier": 3}).to_string(index=False))

Odds multiplier for a one standard deviation increase in each numeric feature:
                feature  one_sd_in_real_units  odds_multiplier
          LOWEST_RATING                  1.04            3.396
                    age                 24.76            1.176
            tavg_mean_c                  5.16            1.107
    MAIN_UNIT_SPANS_045                  4.94            1.052
                    lat                  6.12            1.031
   STRUCTURE_LEN_MT_049                165.45            1.018
  PERCENT_ADT_TRUCK_109                  9.02            1.016
  TRAFFIC_LANES_ON_028A                  1.62            1.012
      DECK_WIDTH_MT_052                 10.79            0.988
                ADT_029              29436.31            0.985
                    lon                 15.03            0.961
       freeze_thaw_days                 38.72            0.959
          prcp_total_mm                436.61            0.949
            tmin_mean_c                

A few things stand out:

* `LOWEST_RATING` is by far the strongest numeric feature. One standard deviation is about one rating point,
  and each point *higher* multiplies the odds of a drop by about 3.4. That sounds backwards at first, and the next
  cell explains it.
* Age matters. Every 25 or so years of age raises the odds of a drop by about 18%.
* Traffic barely matters. Average daily traffic, truck share, and lane count all have multipliers within
  about 2% of 1.0, so once condition and age are known, how busy a bridge is adds little.
* Longer inspection intervals go with lower risk. Bridges on a longer inspection cycle are usually the ones
  already judged to be in good shape, so this reflects the inspectors' own judgment more than anything else.
* The weather features point in mixed directions, for reasons covered in 8.3.

In [24]:
rating_rates = (final_train_df.groupby("LOWEST_RATING")[TARGET]
                .agg(["size", "mean"])
                .rename(columns={"size": "transitions", "mean": "rating_dropped"}))
rating_rates["rating_dropped"] = (rating_rates["rating_dropped"] * 100).round(1)
print("Training data (2010 to 2024 transitions): share of bridges whose rating dropped the next year, by current LOWEST_RATING (%)")
print(rating_rates)

Training data (2010 to 2024 transitions): share of bridges whose rating dropped the next year, by current LOWEST_RATING (%)
               transitions  rating_dropped
LOWEST_RATING                             
0.0                    606             0.0
1.0                    681             1.6
2.0                   3743             1.8
3.0                  13961             2.7
4.0                  66312             2.6
5.0                 254744             2.9
6.0                 540241             3.2
7.0                 719368             4.7
8.0                 103250            13.6
9.0                  17812            23.8


This explains the `LOWEST_RATING` result. Bridges rated 8 or 9 (very good or excellent) have their rating drop
the next year far more often than bridges in the middle of the scale. A bridge rated 9 can only stay at 9 or go
down, and new or freshly repaired bridges tend to settle from 9 to 8 as normal wear shows up. Bridges already rated
low rarely drop further, partly because there is little room left and partly because bridges in poor shape are the
ones that get repaired or closed.

This is an important limit on what the model predicts. The label counts any one point drop, so a slip from 9
(excellent) to 8 (very good) counts exactly the same as a drop from 4 (poor) to 3 (serious). The model's score is
the chance of a rating drop, not the chance of a bridge becoming unsafe. The app needs to describe it that
way. A label that weighs drops by how serious they are would be a good direction for future work.

### 8.2 The biggest effects overall

The chart shows the 20 coefficients with the largest effect in either direction, including the category
indicators. Red bars raise risk and blue bars lower it.

In [25]:
import plotly.express as px

top_effects = coef_table.reindex(coef_table["coefficient"].abs().sort_values(ascending=False).index).head(20)
top_effects = top_effects.sort_values("coefficient")
top_effects["direction"] = np.where(top_effects["coefficient"] > 0, "raises risk", "lowers risk")

fig = px.bar(top_effects, x="coefficient", y="feature", color="direction", orientation="h",
             color_discrete_map={"raises risk": "#e34948", "lowers risk": "#2a78d6"},
             category_orders={"direction": ["raises risk", "lowers risk"]},
             hover_data={"odds_multiplier": ":.2f"},
             labels={"coefficient": "Coefficient (log odds)", "feature": "", "direction": ""},
             title="The 20 largest effects in the final model")
fig.update_yaxes(categoryorder="array", categoryarray=top_effects["feature"].tolist())  # largest raise at top
fig.add_vline(x=0, line_width=1, line_color="#52514e")
fig.update_layout(height=650)
fig.show()

print(top_effects.sort_values("coefficient", ascending=False)[["feature", "coefficient", "odds_multiplier"]]
      .round(3).to_string(index=False))

print("\nState coefficients (each state compared with the others):")
print(coef_table[coef_table["feature"].str.startswith("STATE =")][["feature", "coefficient", "odds_multiplier"]]
      .round(3).to_string(index=False))

                    feature  coefficient  odds_multiplier
       CULVERT_COND_062 = 3        2.546           12.755
          DECK_COND_058 = 2        1.872            6.499
     SCOUR_CRITICAL_113 = 2        1.816            6.149
       CULVERT_COND_062 = 4        1.590            4.905
SUPERSTRUCTURE_COND_059 = 2        1.411            4.100
              LOWEST_RATING        1.223            3.396
  SUBSTRUCTURE_COND_060 = 3        1.214            3.365
          DECK_COND_058 = 1        0.980            2.664
SUPERSTRUCTURE_COND_059 = 3        0.891            2.439
          DECK_COND_058 = 3        0.796            2.218
             OWNER_022 = 70        0.702            2.017
          DECK_COND_058 = 7       -0.698            0.498
  SUBSTRUCTURE_COND_060 = 9       -0.723            0.485
          DECK_COND_058 = N       -0.767            0.464
SUPERSTRUCTURE_COND_059 = 9       -0.791            0.453
          DECK_COND_058 = 9       -0.840            0.432
       CULVERT

The strongest effects are almost all about current condition:

* Components already in poor shape raise risk. A culvert rated 3 (serious) has about 13 times the odds of a
  further drop, a deck rated 2 about 6.5 times, a superstructure rated 2 about 4 times, and a substructure
  rated 3 about 3.4 times.
* Components in good shape lower risk on their own: culverts rated 6 to 9 (down to about 0.3 times the odds), and
  decks, superstructures, and substructures rated 9 (about 0.4 to 0.5 times). This works together with the
  `LOWEST_RATING` effect above; see 8.3 for why these should be read as a group.
* A scour critical rating of 2 (a bridge whose foundation is vulnerable to being washed out by streams) multiplies
  the odds by about 6.
* Washington has lower risk than the other states even after everything else is accounted for (about 0.6 times the
  odds), which matches its low deterioration rate in the EDA. California, New York, and Michigan are the highest,
  at about 1.4 times.

So the answer to research question 2 is that a bridge's current condition is by far the strongest predictor,
followed by age. Traffic, size, and weather matter much less.


### 8.3 A caution: overlapping features

Some features carry nearly the same information. The five weather columns are a clear case: the three temperature
columns are almost the same measurement, and freeze and thaw days move closely with them.

In [26]:
print("Correlation between the weather features (training data):")
print(final_train_df[WEATHER_COLS].corr().round(2))

Correlation between the weather features (training data):


                  tmax_mean_c  tmin_mean_c  tavg_mean_c  prcp_total_mm  freeze_thaw_days
tmax_mean_c              1.00         0.94         0.98          -0.14             -0.81
tmin_mean_c              0.94         1.00         0.98           0.08             -0.85
tavg_mean_c              0.98         0.98         1.00          -0.03             -0.84
prcp_total_mm           -0.14         0.08        -0.03           1.00              0.02
freeze_thaw_days        -0.81        -0.85        -0.84           0.02              1.00


When features overlap this much, the model can split their shared effect between them in odd ways. That is why
the individual weather coefficients point in mixed directions (for example, more freeze and thaw days shows up as
slightly *lower* risk on its own, while a higher average temperature shows up as higher risk). The same thing
happens with `LOWEST_RATING` and the component condition ratings it is calculated from.

The fair way to read overlapping features is as a group, not one coefficient at a time. That is also why Section 6
judged the weather data by adding and removing the whole group and checking the test scores, and why none of the
individual weather coefficients should be read as a finding about climate.


### 8.4 Explaining a single bridge's score

The app needs to explain why a particular bridge got its score. For Logistic Regression this can be done exactly.
After preprocessing, each feature's contribution is its coefficient times its (scaled or 0/1) value. The
contributions plus the model's intercept add up to the bridge's score on the log odds scale, which is then turned
into a probability. So the features with the largest contributions are the reasons behind that bridge's score.

The function below lists the biggest contributions for one bridge, and checks that they really add up to the
model's own score.

In [27]:
def explain_bridge(row, top_n=6):
    """Largest contributions to one bridge's risk score, from the final model."""
    x = final_preprocess.transform(row[FINAL_FEATURE_COLS])
    x = x.toarray()[0] if hasattr(x, "toarray") else np.asarray(x)[0]
    contributions = x * final_clf.coef_[0]

    # The contributions plus the intercept should equal the model's own log odds score
    log_odds = contributions.sum() + final_clf.intercept_[0]
    assert np.isclose(log_odds, final_pipeline.decision_function(row[FINAL_FEATURE_COLS])[0])

    parts = pd.DataFrame({"feature": coef_table["feature"], "contribution": contributions})
    parts = parts[parts["contribution"] != 0]
    parts = parts.reindex(parts["contribution"].abs().sort_values(ascending=False).index).head(top_n)

    # Show numeric features with the bridge's actual value, e.g. "age = 13"
    for col in FINAL_NUMERIC_COLS:
        parts.loc[parts["feature"] == col, "feature"] = f"{col} = {row[col].iloc[0]:g}"
    parts["effect"] = np.where(parts["contribution"] > 0, "raises risk", "lowers risk")
    return parts.round(3).reset_index(drop=True)


test_ranked = test_df.assign(risk=final_test_proba)

# A typical high risk example: the highest scoring test bridge that really did drop, among bridges up to 100 years old
high_risk = (test_ranked[(test_ranked[TARGET] == 1) & (test_ranked["age"] <= 100)]
             .sort_values("risk", ascending=False).head(1))
low_risk = test_ranked.sort_values("risk").head(1)

for label, example in [("High risk example", high_risk), ("Low risk example", low_risk)]:
    r = example.iloc[0]
    print(f"{label}: bridge {r[KEY]} in {r['STATE']}, age {r['age']}, LOWEST_RATING {r['LOWEST_RATING']}")
    print(f"  Predicted chance of a rating drop: {r['risk']:.1%}   Actually dropped in 2025: {'yes' if r[TARGET] else 'no'}")
    print(explain_bridge(example).to_string(index=False))
    print()

High risk example: bridge 000000002204490 in NY, age 5.0, LOWEST_RATING 9.0
  Predicted chance of a rating drop: 96.1%   Actually dropped in 2025: yes
                    feature  contribution      effect
          LOWEST_RATING = 9         3.154 raises risk
          DECK_COND_058 = 9        -0.840 lowers risk
SUPERSTRUCTURE_COND_059 = 9        -0.791 lowers risk
  SUBSTRUCTURE_COND_060 = 9        -0.723 lowers risk
       CULVERT_COND_062 = N         0.419 raises risk
   STRUCTURE_TYPE_043B = 10         0.381 raises risk

Low risk example: bridge 180570C00235001 in TX, age 79.0, LOWEST_RATING 0.0
  Predicted chance of a rating drop: 0.1%   Actually dropped in 2025: no
                  feature  contribution      effect
        LOWEST_RATING = 0        -7.387 lowers risk
        DECK_COND_058 = N        -0.767 lowers risk
SUBSTRUCTURE_COND_060 = N         0.353 raises risk
   SCOUR_CRITICAL_113 = 8        -0.334 lowers risk
                 age = 79         0.233 raises risk
 STRUCTUR

This is the explanation the app will show on its Bridge Lookup page: the bridge's chance of a rating drop, plus
its biggest reasons in plain terms.

Both examples come down to current condition. The high risk bridge is 5 years old and rated 9 across the board.
Its `LOWEST_RATING` of 9 pushes its risk up strongly (it sits at the top of the scale, where drops are most
common), while its individual component ratings of 9 pull the risk partway back down. That is the overlap from
8.3 in action, which is why the app should show the reasons together rather than one at a time. The low risk
bridge has a `LOWEST_RATING` of 0 (failed), so its rating cannot drop any further, and the model gives it almost
no chance of a drop.

The next step saves the final model so the app can load it.